# ColonyNet - Unified Training (UNet++)

Single-stage training on `trainable_pool` with instance metrics and ETA visualization.
Notebook mirrors `train_unified_b3.ipynb` but uses `UNet++`.

In [ ]:
import importlib.util
import subprocess
import sys

# Ensure required packages in active notebook kernel.
_required = {
    'cv2': 'opencv-python',
    'skimage': 'scikit-image',
    'segmentation_models_pytorch': 'segmentation-models-pytorch',
    'timm': 'timm',
}
_missing = [pip_name for mod_name, pip_name in _required.items() if importlib.util.find_spec(mod_name) is None]
if _missing:
    print('Installing missing packages into kernel:', _missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', *_missing])

import os, time, math, yaml
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import display

from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed, find_boundaries
import segmentation_models_pytorch as smp

from colonyseg.utils import set_seed, ensure_dir
from colonyseg.data.datasets import ImageInstancesDataset, split_ids
from colonyseg.data.transforms import build_train_tf, build_val_tf
from colonyseg.metrics.instance_metrics import instance_scores


In [ ]:
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


In [ ]:
def load_yaml(path: str):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

cfg_path = 'configs/train_trainable_pool_mit_b3_edges_safe.yaml'
cfg = load_yaml(cfg_path)

target_cfg = cfg.get('targets', None)

base_run_name = str(cfg.get('run_name', 'colony_trainable_pool_mit_b3_edges_safe'))
cfg['run_name'] = base_run_name.replace('mit_b3', 'unetpp')
if cfg['run_name'] == base_run_name:
    cfg['run_name'] = base_run_name + '_unetpp'

cfg.setdefault('model', {})
cfg['model']['encoder_name'] = cfg['model'].get('encoder_name', 'timm-efficientnet-b3')
cfg['model']['encoder_weights'] = cfg['model'].get('encoder_weights', 'imagenet')

cfg

In [ ]:
set_seed(int(cfg.get('seed', 42)))

run_dir = os.path.join('runs', cfg['run_name'])
ensure_dir(run_dir)

img_size = int(cfg['data']['img_size'])
out_stride = int(cfg['data']['out_stride'])
train_tf = build_train_tf(img_size)
val_tf = build_val_tf(img_size)

all_imgs = sorted([p for p in os.listdir(cfg['data']['train_images']) if p.lower().endswith(('.png','.jpg','.jpeg','.tif','.tiff','.bmp'))])
all_ids = [os.path.splitext(p)[0] for p in all_imgs]
train_ids, val_ids = split_ids(all_ids, float(cfg['data']['val_split']), seed=int(cfg.get('seed', 42)))

train_ds = ImageInstancesDataset(
    cfg['data']['train_images'], cfg['data']['train_instances'],
    transform=train_tf, img_size=img_size, out_stride=out_stride, ids=train_ids, target_cfg=target_cfg
)
val_ds = ImageInstancesDataset(
    cfg['data']['val_images'], cfg['data']['val_instances'],
    transform=val_tf, img_size=img_size, out_stride=out_stride, ids=val_ids, target_cfg=target_cfg
)

train_loader = DataLoader(train_ds, batch_size=int(cfg['train']['batch_size']), shuffle=True,
                          num_workers=int(cfg['train']['num_workers']), pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                        num_workers=max(1, int(cfg['train']['num_workers'])//2), pin_memory=True)

print('train:', len(train_ds), 'val:', len(val_ds))


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
amp_enabled = bool(cfg['train']['amp']) and device == 'cuda'

encoder_name = str(cfg['model'].get('encoder_name', 'timm-efficientnet-b3'))
encoder_weights = cfg['model'].get('encoder_weights', 'imagenet')

model = smp.UnetPlusPlus(
    encoder_name=encoder_name,
    encoder_weights=encoder_weights,
    in_channels=3,
    classes=1,
    activation=None,
).to(device)

lr = float(cfg['train'].get('lr_head', cfg['train'].get('lr', 3e-4)))
wd = float(cfg['train']['weight_decay'])
optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

scaler = torch.amp.GradScaler(device, enabled=amp_enabled)

print('device:', device)
print('encoder:', encoder_name)
print('amp_enabled:', amp_enabled)

In [ ]:
def dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(2, 3))
    union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    return 1.0 - ((2.0 * inter + eps) / (union + eps)).mean()


def sem_to_instances(sem_p, t_sem=0.5, min_distance=3, area_min=4, area_max=25000):
    binary = (sem_p >= t_sem).astype(np.uint8)
    if binary.sum() == 0:
        return np.zeros_like(binary, dtype=np.int32)

    dist = ndi.distance_transform_edt(binary)
    peaks = peak_local_max(dist, labels=binary, min_distance=max(1, int(min_distance)))

    markers = np.zeros_like(binary, dtype=np.int32)
    for i, (yy, xx) in enumerate(peaks, start=1):
        markers[yy, xx] = i

    if markers.max() == 0:
        return np.zeros_like(binary, dtype=np.int32)

    labels = watershed(-dist, markers, mask=binary.astype(bool)).astype(np.int32)

    # Drop too small/large instances and reindex consecutively.
    out = np.zeros_like(labels, dtype=np.int32)
    nxt = 1
    for v in np.unique(labels):
        if v == 0:
            continue
        area = int((labels == v).sum())
        if area < int(area_min):
            continue
        if int(area_max) > 0 and area > int(area_max):
            continue
        out[labels == v] = nxt
        nxt += 1

    return out


epochs = int(cfg['train']['epochs'])
best_f1 = -1.0

iou_thr = float(cfg['train'].get('iou_thr', 0.5))
t_sem = float(cfg['post'].get('t_sem', 0.5))
min_distance = int(cfg['post'].get('min_distance', 3))
area_min = int(cfg['post'].get('area_min', 4))
area_max = int(cfg['post'].get('area_max', 25000))

history = {
    'train_loss': [],
    'val_f1': [],
    'val_merge': [],
    'val_split': [],
    'val_count_err': [],
    'epoch_time': [],
}

plot_handle = None

for epoch in range(1, epochs + 1):
    epoch_start = time.time()

    model.train()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs} [train]')
    loss_sum = 0.0
    n_batches = 0

    for batch in pbar:
        x = batch['image'].to(device, non_blocking=True)
        y_sem = batch['y_sem'].to(device, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device, enabled=amp_enabled):
            logits = model(x)
            if logits.shape[-2:] != y_sem.shape[-2:]:
                logits = F.interpolate(logits, size=y_sem.shape[-2:], mode='bilinear', align_corners=False)

            loss_bce = F.binary_cross_entropy_with_logits(logits, y_sem)
            loss_dice = dice_loss_from_logits(logits, y_sem)
            loss = loss_bce + loss_dice

        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()

        loss_sum += float(loss.item())
        n_batches += 1
        pbar.set_postfix(
            loss='{:.4f}'.format(float(loss.item())),
            bce='{:.4f}'.format(float(loss_bce.item())),
            dice='{:.4f}'.format(float(loss_dice.item())),
        )

    train_loss = loss_sum / max(1, n_batches)

    model.eval()
    metrics = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch}/{epochs} [val]'):
            x = batch['image'].to(device, non_blocking=True)
            gt_inst = batch['instances'].cpu().numpy()[0]

            logits = model(x)
            sem_p = torch.sigmoid(logits).cpu().numpy()[0, 0]

            pr_labels = sem_to_instances(
                sem_p,
                t_sem=t_sem,
                min_distance=min_distance,
                area_min=area_min,
                area_max=area_max,
            )

            out_h, out_w = sem_p.shape
            gt_small = cv2.resize(gt_inst.astype(np.int32), (out_w, out_h), interpolation=cv2.INTER_NEAREST)
            m = instance_scores(gt_small, pr_labels, iou_thr=iou_thr)
            metrics.append(m)

    mean_f1 = float(np.mean([m['f1'] for m in metrics])) if metrics else 0.0
    mean_mer = float(np.mean([m['merge'] for m in metrics])) if metrics else 0.0
    mean_spl = float(np.mean([m['split'] for m in metrics])) if metrics else 0.0
    mean_cnt = float(np.mean([m['count_err'] for m in metrics])) if metrics else 0.0

    last_path = os.path.join(run_dir, 'last.pt')
    torch.save({'epoch': epoch, 'model': model.state_dict(), 'cfg': cfg}, last_path)
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_path = os.path.join(run_dir, 'best.pt')
        torch.save({'epoch': epoch, 'model': model.state_dict(), 'cfg': cfg}, best_path)

    epoch_time = time.time() - epoch_start
    history['train_loss'].append(train_loss)
    history['val_f1'].append(mean_f1)
    history['val_merge'].append(mean_mer)
    history['val_split'].append(mean_spl)
    history['val_count_err'].append(mean_cnt)
    history['epoch_time'].append(epoch_time)

    avg_epoch = float(np.mean(history['epoch_time'][-5:]))
    eta_sec = avg_epoch * (epochs - epoch)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history['train_loss'], label='train_loss')
    ax[0].set_title('Train Loss')
    ax[0].legend()
    ax[1].plot(history['val_f1'], label='val_f1')
    ax[1].set_title('Val F1')
    ax[1].legend()
    if plot_handle is None:
        plot_handle = display(fig, display_id=True)
    else:
        plot_handle.update(fig)
    plt.close(fig)

    print('Epoch {}/{} | train_loss={:.4f} | val_f1={:.4f} | merge={:.3f} | split={:.3f} | count_err={:.3f}'.format(
        epoch, epochs, train_loss, mean_f1, mean_mer, mean_spl, mean_cnt
    ))
    print('Epoch time: {:.1f}s | Avg (last 5): {:.1f}s | ETA: {:.1f} min'.format(
        epoch_time, avg_epoch, eta_sec / 60.0
    ))

In [ ]:
# === paths ===
ckpt_path = os.path.join(run_dir, 'best.pt')
img_path  = 'IMG_4377.jpg'

# === load model ===
ckpt = torch.load(ckpt_path, map_location='cpu')
cfg = ckpt['cfg']

encoder_name = str(cfg.get('model', {}).get('encoder_name', 'timm-efficientnet-b3'))
encoder_weights = cfg.get('model', {}).get('encoder_weights', 'imagenet')

model = smp.UnetPlusPlus(
    encoder_name=encoder_name,
    encoder_weights=encoder_weights,
    in_channels=3,
    classes=1,
    activation=None,
).to(device)
model.load_state_dict(ckpt['model'], strict=True)
model.eval()

# === petri crop helpers (same logic as tool) ===
def detect_petri_circle(img_rgb, min_r_frac=0.35, max_r_frac=0.55, center_tol=0.25):
    h, w = img_rgb.shape[:2]
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (9, 9), 2)

    min_r = int(min(h, w) * min_r_frac)
    max_r = int(min(h, w) * max_r_frac)

    circles = cv2.HoughCircles(
        gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=min(h, w)//2,
        param1=100, param2=30, minRadius=min_r, maxRadius=max_r
    )
    if circles is not None:
        circles = np.round(circles[0]).astype(int)
        cx, cy, r = circles[np.argmax(circles[:, 2])]
    else:
        edges = cv2.Canny(gray, 50, 150)
        cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            return None
        cnt = max(cnts, key=cv2.contourArea)
        (cx_f, cy_f), r_f = cv2.minEnclosingCircle(cnt)
        cx, cy, r = int(cx_f), int(cy_f), int(r_f)

    if r < min_r or r > max_r:
        return None
    cx0, cy0 = w // 2, h // 2
    max_off = center_tol * min(h, w)
    if ((cx - cx0) ** 2 + (cy - cy0) ** 2) ** 0.5 > max_off:
        return None
    return cx, cy, r


def crop_petri(img_rgb, pad=0.02, mask_outside=True):
    h, w = img_rgb.shape[:2]
    circ = detect_petri_circle(img_rgb)
    if circ is None:
        return img_rgb, None, 'no_circle'
    cx, cy, r = circ
    r = int(r * (1.0 + pad))

    x1, y1 = max(0, cx - r), max(0, cy - r)
    x2, y2 = min(w, cx + r), min(h, cy + r)

    crop = img_rgb[y1:y2, x1:x2].copy()

    if mask_outside:
        yy, xx = np.ogrid[y1:y2, x1:x2]
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= (r * r)
        crop[~mask] = 0

    return crop, (cx, cy, r, x1, y1, x2, y2), 'cropped'


def overlay_boundaries(img, lbl):
    out = img.copy()
    b = find_boundaries(lbl, mode='outer')
    out[b] = (0, 255, 0)
    return out


# === load image + petri crop ===
img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

img_petri, meta, status = crop_petri(img_rgb, pad=0.02, mask_outside=True)

# === resize to model input ===
img_rs = cv2.resize(
    img_petri,
    (cfg['data']['img_size'], cfg['data']['img_size']),
    interpolation=cv2.INTER_AREA,
)

# === inference ===
x = torch.from_numpy(img_rs).float().permute(2, 0, 1) / 255.0
x = x.unsqueeze(0).to(device)

with torch.no_grad():
    sem_p = torch.sigmoid(model(x)).cpu().numpy()[0, 0]

labels_up = sem_to_instances(
    sem_p,
    t_sem=float(cfg.get('post', {}).get('t_sem', 0.5)),
    min_distance=int(cfg.get('post', {}).get('min_distance', 3)),
    area_min=int(cfg.get('post', {}).get('area_min', 4)),
    area_max=int(cfg.get('post', {}).get('area_max', 25000)),
)

if labels_up.shape[:2] != img_rs.shape[:2]:
    labels_up = cv2.resize(labels_up.astype(np.int32), (img_rs.shape[1], img_rs.shape[0]), interpolation=cv2.INTER_NEAREST)

# === visualize ===
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].set_title('Original')
ax[0].imshow(img_rgb)
ax[0].axis('off')

ax[1].set_title(f'Petri-cropped ({status})')
ax[1].imshow(img_petri)
ax[1].axis('off')

ax[2].set_title('Pred boundaries')
ax[2].imshow(overlay_boundaries(img_rs, labels_up))
ax[2].axis('off')
plt.show()

In [ ]:
# === MLFLOW AUTO LOGGING ===
import os
import json
from pathlib import Path

try:
    import mlflow
except Exception as exc:
    print(f"[warn] mlflow is unavailable: {exc}")
else:
    tracking_uri = os.getenv('MLFLOW_TRACKING_URI', 'http://127.0.0.1:5000')
    experiment = os.getenv('MLFLOW_EXPERIMENT', 'colony_models')
    run_name = os.getenv('MLFLOW_RUN_NAME', f"{cfg.get('run_name', 'unetpp_unified')}_notebook")

    def _flatten(d, prefix=''):
        out = {}
        if not isinstance(d, dict):
            return out
        for k, v in d.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, dict):
                out.update(_flatten(v, key))
            elif isinstance(v, (list, tuple)):
                out[key] = ','.join(str(x) for x in v)[:250]
            elif v is not None:
                out[key] = str(v)[:250]
        return out

    run_path = Path(run_dir)
    run_path.mkdir(parents=True, exist_ok=True)

    hist = history if isinstance(globals().get('history'), dict) else {}
    history_path = run_path / 'history_unetpp_notebook.json'
    with history_path.open('w', encoding='utf-8') as f:
        json.dump(hist, f, ensure_ascii=False, indent=2)

    history_plot_path = run_path / 'history_unetpp_notebook.png'
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(hist.get('train_loss', []), label='train_loss')
        ax[0].set_title('Train Loss')
        ax[0].legend()
        ax[1].plot(hist.get('val_f1', []), label='val_f1')
        ax[1].set_title('Val F1')
        ax[1].legend()
        fig.tight_layout()
        fig.savefig(history_plot_path, dpi=150)
        plt.close(fig)
    except Exception as exc:
        print(f"[warn] history plot skipped: {exc}")
        history_plot_path = None

    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(experiment)
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({
            'pipeline': 'unetpp_one_stage_notebook',
            'model_name': 'UnetPlusPlus',
            'encoder_name': str(cfg.get('model', {}).get('encoder_name', '')),
        })

        mlflow.log_params(_flatten(cfg))
        mlflow.log_param('run_dir', str(run_path))

        for idx, val in enumerate(hist.get('train_loss', []), start=1):
            mlflow.log_metric('train_loss', float(val), step=idx)
        for idx, val in enumerate(hist.get('val_f1', []), start=1):
            mlflow.log_metric('val_f1', float(val), step=idx)
        for idx, val in enumerate(hist.get('val_merge', []), start=1):
            mlflow.log_metric('val_merge', float(val), step=idx)
        for idx, val in enumerate(hist.get('val_split', []), start=1):
            mlflow.log_metric('val_split', float(val), step=idx)
        for idx, val in enumerate(hist.get('val_count_err', []), start=1):
            mlflow.log_metric('val_count_err', float(val), step=idx)

        if hist.get('val_f1'):
            mlflow.log_metric('best_val_f1', float(max(hist['val_f1'])), step=len(hist['val_f1']))

        artifact_paths = [
            run_path / 'best.pt',
            run_path / 'last.pt',
            history_path,
        ]
        if history_plot_path is not None:
            artifact_paths.append(history_plot_path)

        for p in artifact_paths:
            if p.exists():
                mlflow.log_artifact(str(p), artifact_path='notebook_unetpp')

    print('MLflow logging complete:', run_name)
